In [ ]:
# ==============================================================================
# Squared Sense World Model — Master Kaggle Pipeline
# Dual NVIDIA Tesla T4 GPUs (2x 16 GB VRAM)
# ==============================================================================
import os, glob

# Locate Raw MCA Region Dataset (484 region files)
candidates = [
    '/kaggle/input/datasets/kroissant7/raw-mca-files-484-world-blob/raw_regions',
    '/kaggle/input/raw-mca-files-484-world-blob/raw_regions',
    '/kaggle/input/datasets/kroissant7/raw-mca-files-484-world-blob',
    '/kaggle/input/raw-mca-files-484-world-blob',
]
RAW_REGIONS_DIR = None
for cand in candidates:
    if os.path.exists(cand) and glob.glob(os.path.join(cand, '*.mca')):
        RAW_REGIONS_DIR = cand
        break

if RAW_REGIONS_DIR is None:
    all_mcas = glob.glob('/kaggle/input/**/*.mca', recursive=True)
    if all_mcas:
        RAW_REGIONS_DIR = os.path.dirname(all_mcas[0])

print('Locked RAW_REGIONS_DIR:', RAW_REGIONS_DIR)
mca_count = len(glob.glob(os.path.join(RAW_REGIONS_DIR, '*.mca'))) if RAW_REGIONS_DIR else 0
print(f'Total Raw MCA files found: {mca_count}')

In [ ]:
# Clone Squared-Sense-World-Model repository
!git clone https://github.com/Squared-Sense/Squared-Sense-World-Model.git /kaggle/working/Squared-Sense-World-Model
%cd /kaggle/working/Squared-Sense-World-Model
!git fetch origin world && git checkout world && git pull origin world
!git status

In [ ]:
!pip install -q uv
%env UV_PROJECT_ENVIRONMENT=/tmp/.venv

In [ ]:
!cd /kaggle/working/Squared-Sense-World-Model && uv sync --no-dev

In [ ]:
!uv pip install --python $(which python3) flax optax nbtlib psutil einops matplotlib pandas scikit-learn

In [ ]:
import os, glob, sys

REPO_ROOT = '/kaggle/working/Squared-Sense-World-Model'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.environ['PYTHONPATH'] = REPO_ROOT
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'

# ==============================================================================
# Ephemeral Scratchpad Storage for Dense Voxel Arrays
# /tmp has ~73 GB of free ephemeral space on Kaggle containers.
# Placing baked memmaps (~19.2 GB) in /tmp ensures they DO NOT count toward
# the 20 GB output persistence limit of /kaggle/working/.
# ==============================================================================
DATA_ROOT = '/tmp/squared_sense_data'
BAKED_DIR = os.path.join(DATA_ROOT, 'baked_regions')
EPISODES_DIR = os.path.join(DATA_ROOT, 'episodes')
os.makedirs(BAKED_DIR, exist_ok=True)
os.makedirs(EPISODES_DIR, exist_ok=True)
os.environ['BAKED_DIR'] = BAKED_DIR
os.environ['EPISODES_DIR'] = EPISODES_DIR

print('Locked REPO_ROOT:                 ', REPO_ROOT)
print('Baked Regions Dir (in /tmp):      ', BAKED_DIR)
print('Episodes Dir (in /tmp):           ', EPISODES_DIR)

In [ ]:
!uv add jax[cuda13]

In [ ]:
import jax

print('Backend:', jax.default_backend())
print('Devices:', jax.devices())

## Step 0: On-Kaggle Data Preparation Pipeline
Bakes raw Minecraft Anvil (`.mca`) files into zero-copy binary memmap arrays (`.mmap`), indexes terrain spawns, and rolls out standardized `.npz` trajectory episodes.

- **Baking Limit**: Out of 484 raw MCA files (~4 GB), we bake **100 central regions** closest to world origin `(0, 0)`. This guarantees a contiguous, dense $10 \times 10$ terrain cluster around spawn.
- **Ephemeral Scratch Storage (`/tmp`)**: All dense `.mmap` files (~19.2 GB) and `.npz` episodes are placed in `/tmp/squared_sense_data`. This bypasses Kaggle's 20 GB `/kaggle/working` output limit so that all checkpoints and final artifacts persist without disk pressure.

In [ ]:
# Step 0A: Bake 100 central MCA region files into memmap arrays in ephemeral /tmp storage
# mca_baker sorts regions by Euclidean distance from (0, 0) so the 100 closest are selected
!uv run python -m env.mca_baker \
  --input {RAW_REGIONS_DIR} \
  --output {BAKED_DIR} \
  --limit 100

In [ ]:
# Step 0B: Index valid spawn points (60% surface, 25% subterranean, 15% steep)
!uv run python -m env.spawn_indexer \
  --dir {BAKED_DIR}

In [ ]:
# Step 0C: Generate 100 trajectory episodes (5,000 frames) across baked terrain
!uv run python -m env.record_episodes \
  --region_dir {BAKED_DIR} \
  --out_dir {EPISODES_DIR} \
  --num_episodes 100 \
  --ep_len 50 \
  --policy heuristic

In [ ]:
# Verify Data Pipeline Output in Ephemeral Scratchpad
import glob, os

mmaps = glob.glob(os.path.join(BAKED_DIR, '*.mmap'))
spawns = glob.glob(os.path.join(BAKED_DIR, '*.spawns.npz'))
eps = glob.glob(os.path.join(EPISODES_DIR, '*.npz'))

mmap_mb = sum(os.path.getsize(f) for f in mmaps) / 1e6
ep_mb = sum(os.path.getsize(f) for f in eps) / 1e6

print(f'Baked Memmaps : {len(mmaps)} files ({mmap_mb:.1f} MB in {BAKED_DIR})')
print(f'Spawn Indexes : {len(spawns)} files')
print(f'Recorded Eps  : {len(eps)} files ({ep_mb:.1f} MB in {EPISODES_DIR})')

## Step 1A: Stage 1A — M2 Static Spatial JEPA Pretraining (80 Epochs)
Trains the Multi-LOD Concentric Shell Transformer Backbone with 3D block-masking and target EMA encoder.
*(If you already have `checkpoints/m2_spatial_jepa/best.pkl`, you can skip to Step 2).* 

In [ ]:
# Stage 1A Spatial JEPA Training (80 Epochs)
!uv run python -m train.train_spatial_jepa \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --save_dir checkpoints/m2_spatial_jepa \
  --epochs 80 \
  --batch_size 32 \
  --lr 3e-4 \
  --warmup_epochs 10 \
  --log_every 200

In [ ]:
# Plot Stage 1A Spatial JEPA Loss, Collapse Prevention & Representation Rank
import json, os
import matplotlib.pyplot as plt
import numpy as np

metrics_path = 'checkpoints/m2_spatial_jepa/metrics.jsonl'

if os.path.exists(metrics_path):
    with open(metrics_path, 'r', encoding='utf-8') as f:
        rows = [json.loads(line) for line in f if line.strip()]

    epochs = [r['epoch'] for r in rows]
    train_loss = [r['train_loss'] for r in rows]
    val_loss = [r['val_loss'] for r in rows]
    pred_loss = [r.get('train_pred_loss', r['train_loss']) for r in rows]
    emb_std = [r.get('emb_std', 1.0) for r in rows]
    lr = [r.get('lr', 0.0) for r in rows]
    ema_m = [r.get('ema_momentum', 0.996) for r in rows]

    rankme_epochs = [r['epoch'] for r in rows if r.get('rankme') is not None]
    rankme_vals = [r['rankme'] for r in rows if r.get('rankme') is not None]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.2), dpi=150, facecolor='white')

    # 1. Total Loss & Target Prediction L1
    axes[0].plot(epochs, train_loss, label='Train Total', color='#1f77b4', lw=2)
    axes[0].plot(epochs, val_loss, label='Val Total', color='#d62728', lw=2)
    axes[0].plot(epochs, pred_loss, label='Pred L1', color='#1f77b4', ls='--', alpha=0.7)
    axes[0].set_title('Spatial JEPA Losses (80 Epochs)', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Epoch', fontsize=10)
    axes[0].set_ylabel('Loss', fontsize=10)
    axes[0].legend(fontsize=9, loc='upper right')
    axes[0].grid(True, alpha=0.25)

    # 2. Representation Anti-Collapse: Embedding Std Dev
    axes[1].plot(epochs, emb_std, color='#2ca02c', lw=2, label='Val Emb Std')
    axes[1].axhline(1.0, color='gray', ls=':', lw=1.5, label='VICReg Target (σ=1.0)')
    axes[1].set_title('Representation Variance (Anti-Collapse)', fontsize=11, fontweight='bold')
    axes[1].set_xlabel('Epoch', fontsize=10)
    axes[1].set_ylabel('Embedding Std Dev', fontsize=10)
    axes[1].legend(fontsize=9, loc='lower right')
    axes[1].grid(True, alpha=0.25)

    # 3. Representation Capacity: RankMe
    if rankme_vals:
        axes[2].plot(rankme_epochs, rankme_vals, marker='o', markersize=5, color='#9467bd', lw=2, label='RankMe Rank')
        axes[2].axhline(128.0, color='gray', ls=':', lw=1.5, label='Healthy Dim Target (128/256)')
        axes[2].set_title('Effective Dimension (RankMe)', fontsize=11, fontweight='bold')
        axes[2].set_xlabel('Epoch', fontsize=10)
        axes[2].set_ylabel('Effective Rank', fontsize=10)
        axes[2].legend(fontsize=9, loc='lower right')
        axes[2].grid(True, alpha=0.25)
    else:
        axes[2].text(0.5, 0.5, 'RankMe computed periodically', ha='center', va='center', transform=axes[2].transAxes)

    # 4. Learning Rate Schedule & EMA Target Momentum
    ax4_twin = axes[3].twinx()
    l1 = axes[3].plot(epochs, lr, color='#ff7f0e', lw=2, label='Learning Rate')
    l2 = ax4_twin.plot(epochs, ema_m, color='#8c564b', ls='--', lw=1.8, label='EMA Momentum')
    axes[3].set_title('Warmup LR & Target EMA', fontsize=11, fontweight='bold')
    axes[3].set_xlabel('Epoch', fontsize=10)
    axes[3].set_ylabel('Learning Rate', fontsize=10)
    ax4_twin.set_ylabel('EMA Momentum', fontsize=10)
    axes[3].ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    axes[3].legend(lines, labels, fontsize=8.5, loc='center right')
    axes[3].grid(True, alpha=0.25)

    plt.tight_layout()
    os.makedirs('/kaggle/working/paper_figs', exist_ok=True)
    out_fig = '/kaggle/working/paper_figs/m2_spatial_curves.png'
    plt.savefig(out_fig, dpi=300, bbox_inches='tight')
    plt.savefig('/kaggle/working/paper_figs/m2_spatial_curves.pdf', bbox_inches='tight')
    print(f'Stage 1A training curves saved to: {out_fig}')
    plt.show()
else:
    print(f'Notice: {metrics_path} not found. Run Stage 1A training or ensure metrics.jsonl exists in checkpoints.')

In [ ]:
# Visualize DINO-Style 3D Voxel Concentric Shell Token Cosine Similarity Maps (Figure 2)
import os, glob
import matplotlib.pyplot as plt
import numpy as np

from paper_figs.make_figures import plot_3d_patch_similarity
from models.transformer import SpatialJEPA
from train.checkpoint import load_checkpoint

ckpt_path = 'checkpoints/m2_spatial_jepa/best.pkl'
shells_single = None

# Try loading real voxel shells from recorded episodes
ep_files = glob.glob(f'{EPISODES_DIR}/*.npz')
if ep_files:
    try:
        data = np.load(ep_files[0])
        if 'shells' in data:
            shells_single = data['shells'][0]
            print('Loaded real terrain voxel shells from:', os.path.basename(ep_files[0]))
    except Exception as e:
        print(f'Episode load notice: {e}')

backbone = None
if os.path.exists(ckpt_path):
    try:
        import flax.nnx as nnx, jax
        model = SpatialJEPA(num_shells=3, rngs=nnx.Rngs(42))
        meta = load_checkpoint(ckpt_path, model)
        backbone = model
        print(f'Loaded Stage 1A checkpoint (Epoch: {meta.get("epoch", "N/A")}, Val Loss: {meta.get("val_loss", 0.0):.4f})')
    except Exception as e:
        print(f'Using representative feature computation: {e}')

pdf_p, png_p = plot_3d_patch_similarity(
    backbone=backbone,
    shells_single=shells_single,
    query_idx=36,  # Center patch in immediate LOD 0 shell
    out_dir='/kaggle/working/paper_figs',
    dpi=300
)

from IPython.display import Image, display
if os.path.exists(png_p):
    print(f'Figure 2 saved to: {png_p}')
    display(Image(filename=png_p))

In [ ]:
# Archive Stage 1A Weights
import os, shutil
from IPython.display import FileLink

src_m2 = os.path.abspath('checkpoints/m2_spatial_jepa')
if os.path.exists(src_m2):
    zip_m2 = shutil.make_archive('/kaggle/working/m2_spatial_jepa_weights', 'zip', src_m2)
    print(f'Saved M2 archive: {zip_m2} ({os.path.getsize(zip_m2) / 1e6:.1f} MB)')
    FileLink('m2_spatial_jepa_weights.zip')

## Step 2: Stage 1B — Dual-LSTM Temporal JEPA Pretraining (M1 Full)
Trains the `DualLSTMTemporalIntegrator` (~0.72M params) on top of the frozen Stage 1A spatial backbone with forward autoregressive rollout and IDM auxiliary loss.

- **Phase 1 (15 epochs)**: Frozen spatial backbone; optimizes Dual-LSTM on stop-gradient spatial representations.
- **Phase 2 (25 epochs)**: Unfreezes top 2 Transformer blocks + final norm of context encoder with spatial anchor loss on random frames.

In [ ]:
# Stage 1B Dual-LSTM Temporal JEPA Pretraining (M1 Full)
!uv run python -m train.train_temporal_jepa \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --spatial_ckpt checkpoints/m2_spatial_jepa/best.pkl \
  --save_dir checkpoints/m1_full_temporal \
  --phase1_epochs 15 \
  --phase2_epochs 25 \
  --batch_size 8 \
  --clip_len 16 \
  --k_steps 4

In [ ]:
# Inspect Stage 1B Checkpoints & Create Weights Archive
import os, shutil
from IPython.display import FileLink

src_m1 = os.path.abspath('checkpoints/m1_full_temporal')
if os.path.exists(src_m1):
    print('Files in checkpoint dir:', os.listdir(src_m1))
    zip_m1 = shutil.make_archive('/kaggle/working/m1_full_temporal_weights', 'zip', src_m1)
    print(f'Saved M1 archive: {zip_m1} ({os.path.getsize(zip_m1) / 1e6:.1f} MB)')
    FileLink('m1_full_temporal_weights.zip')

In [ ]:
# Plot Stage 1B Temporal JEPA Loss & Training Dynamics
import json, os
import matplotlib.pyplot as plt

jsonl_path = 'checkpoints/m1_full_temporal/metrics.jsonl'
if os.path.exists(jsonl_path):
    with open(jsonl_path) as f:
        rows = [json.loads(line) for line in f if line.strip()]

    train_loss = [r['train_loss'] for r in rows]
    val_loss = [r['val_loss'] for r in rows]
    rollout_l1 = [r.get('rollout_l1', 0.0) for r in rows]
    idm_loss = [r.get('idm_loss', 0.0) for r in rows]
    rankme = [r.get('rankme', 0.0) for r in rows]

    fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), facecolor='white')

    axes[0].plot(train_loss, label='Train Loss', lw=2)
    axes[0].plot(val_loss, label='Val Loss', lw=2)
    axes[0].set_title('Temporal JEPA Total Loss')
    axes[0].set_xlabel('Epoch / Step')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(rollout_l1, color='tab:orange', label='Rollout L1', lw=2)
    axes[1].plot(idm_loss, color='tab:red', label='IDM Loss', lw=2)
    axes[1].set_title('Rollout L1 & Inverse Dynamics Loss')
    axes[1].set_xlabel('Epoch / Step')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(rankme, marker='o', color='tab:purple', lw=2)
    axes[2].set_title('Context Effective Rank (RankMe)')
    axes[2].set_xlabel('Epoch / Step')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    os.makedirs('/kaggle/working/paper_figs', exist_ok=True)
    plt.savefig('/kaggle/working/paper_figs/m1_temporal_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

## Step 3: Parameter-Matched Baseline Pretraining
Trains single-LOD baselines and generative 3D-MAE for parameter-matched comparison against M2 (~4.65M) and M1 (~5.37M).

In [ ]:
# Baseline b1_lod0: Single-LOD 0 JEPA (64 tokens, ~4.65M params)
!uv run python -m train.train_baselines --model b1_lod0 \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --save_dir checkpoints/baselines/b1_lod0 --epochs 80 --batch_size 32

In [ ]:
# Baseline b2_lod2: Single-LOD 2 JEPA (64 tokens, ~4.65M params)
!uv run python -m train.train_baselines --model b2_lod2 \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --save_dir checkpoints/baselines/b2_lod2 --epochs 80 --batch_size 32

In [ ]:
# Baseline b3_mae: Generative 3D-MAE (~4.80M params)
!uv run python -m train.train_baselines --model b3_mae \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --save_dir checkpoints/baselines/b3_mae --epochs 80 --batch_size 32

In [ ]:
# Archive Baseline Weights
src_b = os.path.abspath('checkpoints/baselines')
if os.path.exists(src_b):
    zip_b = shutil.make_archive('/kaggle/working/baselines_weights', 'zip', src_b)
    print(f'Saved baselines archive: {zip_b} ({os.path.getsize(zip_b) / 1e6:.1f} MB)')
    FileLink('baselines_weights.zip')

## Step 4: Frozen Representation Diagnostic Probing Suite
Applies lightweight linear and logistic regression probes (Distant Goal $R^2$ / Cosine Sim, Local Hazard F1 / AUROC, Velocity $R^2$, Occlusion Memory F1, RankMe, Multi-step rollout) on frozen representations across 3 random seeds.

In [ ]:
# Probe M1 Full (Fused Spatial + Temporal Representation)
!uv run python -m eval.probe_suite \
  --model_type m1_full \
  --ckpt checkpoints/m1_full_temporal/spatial_finetuned_best.pkl \
  --temporal_ckpt checkpoints/m1_full_temporal/temporal_best.pkl \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --out_json checkpoints/m1_full_temporal/probe_results.json

# Probe M2 Static (Spatial JEPA Context Backbone)
!uv run python -m eval.probe_suite \
  --model_type m2_static \
  --ckpt checkpoints/m2_spatial_jepa/best.pkl \
  --data_dir {EPISODES_DIR} \
  --region_dir {BAKED_DIR} \
  --out_json checkpoints/m2_spatial_jepa/probe_results.json

In [ ]:
# Probe Baselines (b1_lod0, b2_lod2, b3_mae) if trained
for b_name in ['b1_lod0', 'b2_lod2', 'b3_mae']:
    ckpt_path = f'checkpoints/baselines/{b_name}/best.pkl'
    if os.path.exists(ckpt_path):
        out_j = f'checkpoints/baselines/{b_name}/probe_results.json'
        !uv run python -m eval.probe_suite \
          --model_type {b_name} \
          --ckpt {ckpt_path} \
          --data_dir {EPISODES_DIR} \
          --region_dir {BAKED_DIR} \
          --out_json {out_j}

In [ ]:
# Synthesize Diagnostic Probing Comparative Summary Table
import json, glob, pandas as pd

probe_files = glob.glob('checkpoints/**/probe_results.json', recursive=True)
records = []
for p in sorted(probe_files):
    with open(p) as f:
        d = json.load(f)
    records.append({
        'Model': d.get('model_type'),
        'Dim': d.get('feature_dim'),
        'Goal R²': f"{d['distant_goal']['r2']:.3f}±{d['distant_goal']['r2_std']:.3f}",
        'Goal CosSim': f"{d['distant_goal']['mean_cosine_sim']:.3f}",
        'Hazard F1': f"{d['local_hazard']['f1']:.3f}±{d['local_hazard']['f1_std']:.3f}",
        'Hazard AUROC': f"{d['local_hazard']['auroc']:.3f}",
        'Vel R²': f"{d['kinematic_and_occlusion']['velocity_r2']:.3f}",
        'Occl F1': f"{d['kinematic_and_occlusion']['occlusion_f1']:.3f}",
        'RankMe': f"{d['multistep_rollout']['rankme']:.1f}",
        'Rollout H4 Cos': f"{d['multistep_rollout'].get('cosine_sim_h4', 0.0):.3f}",
    })

df = pd.DataFrame(records)
print('=== Diagnostic Probing Comparative Summary ===')
display(df) if 'display' in globals() else print(df.to_string(index=False))

## Step 5: Downstream Policy Training (Frozen-Backbone PPO Navigation)
Trains the lightweight `ActorCriticHead` (~50K params) strictly on top of frozen perception representations on MinecraftNavEnv real terrain.

In [ ]:
# Train PPO Navigation Policy on M1 Full (50,000 steps)
!uv run python -m eval.train_ppo_nav \
  --model_type m1_full \
  --ckpt checkpoints/m1_full_temporal/spatial_finetuned_best.pkl \
  --temporal_ckpt checkpoints/m1_full_temporal/temporal_best.pkl \
  --region_dir {BAKED_DIR} \
  --save_dir checkpoints/ppo_m1_full \
  --total_steps 50000

In [ ]:
# Train PPO Navigation Policy on M2 Static (50,000 steps) for comparison
!uv run python -m eval.train_ppo_nav \
  --model_type m2_static \
  --ckpt checkpoints/m2_spatial_jepa/best.pkl \
  --region_dir {BAKED_DIR} \
  --save_dir checkpoints/ppo_m2_static \
  --total_steps 50000

In [ ]:
# Inspect & Archive PPO Policy Checkpoints
src_ppo = os.path.abspath('checkpoints/ppo_m1_full')
if os.path.exists(src_ppo):
    zip_ppo = shutil.make_archive('/kaggle/working/ppo_m1_full_policy', 'zip', src_ppo)
    print(f'Saved PPO policy archive: {zip_ppo} ({os.path.getsize(zip_ppo) / 1e6:.1f} MB)')
    FileLink('ppo_m1_full_policy.zip')

## Step 6: Zero-Shot Navigation Evaluation & Shell Gating Profiling
Evaluates navigation performance: Success Rate (SR %), Success weighted by Path Length (SPL), Waypoint Completion %, Hazard Collision Rate %, and profiles empirical MFLOPs/step and wall-clock Latency (ms/step) with Event-Driven Shell Gating acceleration.

In [ ]:
# Evaluate M1 Full Navigation with Event-Driven Shell Gating (--use_gating)
!uv run python -m eval.eval_navigation \
  --model_type m1_full \
  --backbone_ckpt checkpoints/m1_full_temporal/spatial_finetuned_best.pkl \
  --temporal_ckpt checkpoints/m1_full_temporal/temporal_best.pkl \
  --policy_ckpt checkpoints/ppo_m1_full/policy_best.pkl \
  --region_dir {BAKED_DIR} \
  --use_gating \
  --num_episodes 20 \
  --max_steps 150 \
  --out_json eval/navigation_gating_results_m1.json

# Evaluate M1 Full Dense (Dense inference baseline without gating)
!uv run python -m eval.eval_navigation \
  --model_type m1_full \
  --backbone_ckpt checkpoints/m1_full_temporal/spatial_finetuned_best.pkl \
  --temporal_ckpt checkpoints/m1_full_temporal/temporal_best.pkl \
  --policy_ckpt checkpoints/ppo_m1_full/policy_best.pkl \
  --region_dir {BAKED_DIR} \
  --num_episodes 20 \
  --max_steps 150 \
  --out_json eval/navigation_dense_results_m1.json

In [ ]:
# Evaluate M2 Static for cross-architecture comparison
!uv run python -m eval.eval_navigation \
  --model_type m2_static \
  --backbone_ckpt checkpoints/m2_spatial_jepa/best.pkl \
  --policy_ckpt checkpoints/ppo_m2_static/policy_best.pkl \
  --region_dir {BAKED_DIR} \
  --num_episodes 20 \
  --max_steps 150 \
  --out_json eval/navigation_results_m2.json

In [ ]:
# Navigation & Shell Gating Efficiency Summary
import json, glob, os, pandas as pd

eval_files = sorted(set(glob.glob('eval/navigation_*.json') + glob.glob('results/nav_*.json')))
eval_records = []
for ef in eval_files:
    with open(ef, 'r', encoding='utf-8') as f:
        d = json.load(f)
    eval_records.append({
        'Config': os.path.basename(ef).replace('.json', ''),
        'Gating': d.get('use_gating', False),
        'SR %': f"{d.get('success_rate_pct', 0.0):.1f}%",
        'SPL': f"{d.get('spl', 0.0):.3f}",
        'CHNS': f"{d.get('chns', 0.0):.1f}",
        'Compass %': f"{d.get('compass_visible_pct', 100.0):.1f}%",
        'Waypoint %': f"{d.get('waypoint_completion_pct', 0.0):.1f}%",
        'Hazard %': f"{d.get('hazard_collision_rate_pct', 0.0):.1f}%",
        'MFLOPs/step': f"{d.get('empirical_mflops_per_step', 0.0):.1f}",
        'Latency (ms)': f"{d.get('latency_ms_per_step', 0.0):.2f}",
    })

df_nav = pd.DataFrame(eval_records)
print('=== Zero-Shot Navigation & Gating Summary ===')
display(df_nav) if 'display' in globals() else print(df_nav.to_string(index=False))

## Step 7: Final Paper Vector Figures & Master Artifact Archive
Generates all 4 publication-quality Springer Nature LNEE vector figures (.pdf) and 600 DPI previews (.png) and packages all project outputs into a single download archive.

In [ ]:
# Generate Springer Nature LNEE Vector Figures
!uv run python paper_figs/make_figures.py

# Verify generated figures
!ls -lh paper_figs/

In [ ]:
# Master Artifact Packaging: Checkpoints, Figures, Metrics & JSON Summaries
import shutil, os
from IPython.display import FileLink

export_dir = '/kaggle/working/squared_sense_master_export'
os.makedirs(export_dir, exist_ok=True)

for subdir in ['checkpoints', 'paper_figs', 'eval']:
    src_sub = os.path.join('/kaggle/working/Squared-Sense-World-Model', subdir)
    dst_sub = os.path.join(export_dir, subdir)
    if os.path.exists(src_sub):
        shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)

# Also copy recorded episodes (~150 MB) from /tmp scratchpad for reproducibility
if os.path.exists(EPISODES_DIR):
    shutil.copytree(EPISODES_DIR, os.path.join(export_dir, 'episodes'), dirs_exist_ok=True)

master_zip = shutil.make_archive('/kaggle/working/squared_sense_master_artifacts', 'zip', export_dir)
print(f'Master artifact archive created successfully: {master_zip} ({os.path.getsize(master_zip) / 1e6:.1f} MB)')
FileLink('squared_sense_master_artifacts.zip')

# One-Click GPU Training + Benchmarks + JSON/JSONL Logging + Zip Packaging
%cd /kaggle/working/Squared-Sense-World-Model
!uv add jax[cuda13]
!chmod +x scripts/kaggle_run_all.sh
!BAKED_DIR=/kaggle/input/datasets/algionightly/squared-sense/data/baked_regions EPISODES_DIR=/kaggle/input/datasets/algionightly/squared-sense/data/episodes EVAL_EPISODES=20 SEED=35724444 bash scripts/kaggle_run_all.sh

In [ ]:
# One-Click GPU Training + Benchmarks + JSON/JSONL Logging + Zip Packaging
%cd /kaggle/working/Squared-Sense-World-Model
!uv add jax[cuda13]
!chmod +x scripts/kaggle_run_all.sh
!BAKED_DIR=/kaggle/input/datasets/algionightly/squared-sense/data/baked_regions EPISODES_DIR=/kaggle/input/datasets/algionightly/squared-sense/data/episodes EVAL_EPISODES=20 SEED=35724444 bash scripts/kaggle_run_all.sh

In [ ]:
# Inspect Consolidated JSONL Benchmark Log
import json, os, pandas as pd

jsonl_path = 'results/all_baselines.jsonl'
if os.path.exists(jsonl_path):
    rows = [json.loads(line) for line in open(jsonl_path, encoding='utf-8') if line.strip()]
    nav_rows = [r for r in rows if 'success_rate_pct' in r]
    if nav_rows:
        df_all = pd.DataFrame([{
            'Benchmark': r.get('_file', '').replace('.json', ''),
            'SR %': round(r.get('success_rate_pct', 0.0), 1),
            'SPL': round(r.get('spl', 0.0), 3),
            'CHNS': round(r.get('chns', 0.0), 1),
            'Compass %': round(r.get('compass_visible_pct', 100.0), 1),
            'Hazard %': round(r.get('hazard_collision_rate_pct', 0.0), 1),
            'MFLOPs': round(r.get('empirical_mflops_per_step', 0.0), 1),
        } for r in nav_rows])
        display(df_all) if 'display' in globals() else print(df_all.to_string(index=False))
else:
    print('Run Step 8 cell first to generate results/all_baselines.jsonl')

## Step 9: Auto-Upload Master `.zip` After Training (Unattended)
Choose `AUTO_UPLOAD_MODE` (`'kaggle_dataset'`, `'hf_hub'`, or `'gdrive_rclone'`). Store credentials in **Kaggle Add-ons -> Secrets** so uploads run automatically when training finishes—even if your browser closes. If you use **Save & Run All (Commit)**, `/kaggle/working/squared_sense_master_artifacts.zip` is also saved to the Kaggle Output tab automatically.

In [ ]:
# Auto-Upload Master Zip Archive (Checkpoints + JSON/JSONL Logs + Figures)
import os, json, shutil, subprocess
from IPython.display import FileLink

ZIP_PATH = '/kaggle/working/squared_sense_master_artifacts.zip'
AUTO_UPLOAD_MODE = 'kaggle_dataset'  # Options: 'kaggle_dataset', 'hf_hub', 'gdrive_rclone', 'none'

# Target identifiers (customize before running)
KAGGLE_DATASET_SLUG = 'squared-sense-world-model-artifacts'  # -> <KAGGLE_USERNAME>/<slug>
HF_REPO_ID = 'your-username/squared-sense-artifacts'          # private HF dataset/model repo
GDRIVE_REMOTE_PATH = 'gdrive:SquaredSenseBackups/'            # rclone remote:folder

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

if not os.path.exists(ZIP_PATH):
    print(f'Archive not found at {ZIP_PATH}. Run Step 8 first.')
else:
    print(f'Ready to upload: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.2f} MB)')

    if AUTO_UPLOAD_MODE == 'kaggle_dataset':
        k_user, k_key = get_secret('KAGGLE_USERNAME'), get_secret('KAGGLE_KEY')
        if k_user and k_key:
            os.environ['KAGGLE_USERNAME'] = k_user
            os.environ['KAGGLE_KEY'] = k_key
            bundle_dir = '/tmp/kaggle_dataset_upload'
            os.makedirs(bundle_dir, exist_ok=True)
            shutil.copy2(ZIP_PATH, os.path.join(bundle_dir, os.path.basename(ZIP_PATH)))
            meta = {
                'title': 'Squared Sense World Model Artifacts',
                'id': f'{k_user}/{KAGGLE_DATASET_SLUG}',
                'licenses': [{'name': 'CC0-1.0'}]
            }
            with open(os.path.join(bundle_dir, 'dataset-metadata.json'), 'w') as mf:
                json.dump(meta, mf, indent=2)
            res = subprocess.run(['kaggle', 'datasets', 'version', '-p', bundle_dir, '-m', 'Auto-upload after training', '--dir-mode', 'zip'], capture_output=True, text=True)
            if res.returncode != 0:
                res = subprocess.run(['kaggle', 'datasets', 'create', '-p', bundle_dir, '--dir-mode', 'zip'], capture_output=True, text=True)
            print(res.stdout or res.stderr)
        else:
            print('Set KAGGLE_USERNAME and KAGGLE_KEY in Kaggle Secrets to auto-upload to Kaggle Datasets.')

    elif AUTO_UPLOAD_MODE == 'hf_hub':
        hf_token = get_secret('HF_TOKEN')
        if hf_token:
            subprocess.run(['pip', 'install', '-q', 'huggingface_hub'], check=True)
            from huggingface_hub import HfApi
            api = HfApi(token=hf_token)
            api.create_repo(repo_id=HF_REPO_ID, repo_type='dataset', private=True, exist_ok=True)
            api.upload_file(path_or_fileobj=ZIP_PATH, path_in_repo=os.path.basename(ZIP_PATH), repo_id=HF_REPO_ID, repo_type='dataset')
            print(f'Uploaded to Hugging Face Hub: {HF_REPO_ID}')
        else:
            print('Set HF_TOKEN in Kaggle Secrets to auto-upload to Hugging Face Hub.')

    elif AUTO_UPLOAD_MODE == 'gdrive_rclone':
        rclone_conf = get_secret('RCLONE_CONF')
        if rclone_conf:
            os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
            with open(os.path.expanduser('~/.config/rclone/rclone.conf'), 'w') as rf:
                rf.write(rclone_conf)
            subprocess.run('curl -fsSL https://rclone.org/install.sh | bash', shell=True, check=True)
            subprocess.run(['rclone', 'copy', ZIP_PATH, GDRIVE_REMOTE_PATH, '--progress'], check=True)
            print(f'Uploaded to Google Drive: {GDRIVE_REMOTE_PATH}')
        else:
            print('Set RCLONE_CONF in Kaggle Secrets to auto-upload to Google Drive.')

    FileLink('squared_sense_master_artifacts.zip')